# Colab full-run: primary IPI detector (parallel to local MPS run)

See `docs/DECISIONS.md` D29 for why this exists: local MPS hit two real numerical bugs (AdamW `eps=1e-8` underflow, a silent fp16 load) documented in `docs/ISSUES.md` ISSUE-1/ISSUE-3, and the fp32 correctness fix costs ~4x per-step time on MPS. Running here in parallel, not instead of local — whichever finishes a clean full run first wins.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine).

**Reproducibility note:** `torch.manual_seed()` is now called before model construction (peer-review finding, fixed) so `--seed` covers the classifier head's random init, not just batch order. Exact reproduction across MPS and CUDA is still not guaranteed even with the same seed (different RNG backends, cuDNN algorithm selection) — Step 5 below checks for "comparably healthy," not bit-identical, numbers.

**Checkpoint layout note:** each epoch writes to its own `epoch_N/` subdirectory under the run's checkpoint dir, never overwriting a prior epoch (`docs/ISSUES.md` ISSUE-6 correction) — `save_pretrained()` isn't atomic, so a shared-path overwrite risked corrupting the directory and destroying the last known-good epoch on an interrupted write. This notebook always resolves "the checkpoint" as the highest-numbered `epoch_N/` that exists.

**Torchvision note (ISSUE-7):** Step 3 uninstalls `torchvision` after the pinned-requirements install. Colab ships it pre-installed against Colab's own default torch build; our pinned `torch==2.13.0` leaves that torchvision ABI-incompatible, which crashes `transformers`' model-loading machinery (it transitively imports `torchvision.io` while resolving `DebertaV2ForSequenceClassification`, unrelated to anything we actually use). This project has zero torchvision usage — removing it is the fix, not a workaround.

**Step 1** below needs the *current* local working tree's code (fp32 pin, AdamW eps fix, per-epoch checkpointing, the seeding fix, the epoch-subdir fix, and the sys.path portability fix are all local changes — confirm the pushed branch actually includes all of these before relying on Step 5's comparison). Either:
- (a) the operator pushes the current branch to `origin` first and this notebook clones it, or
- (b) the operator uploads a zip of the working tree directly (see the commented-out alternative in Step 1).

`data/processed/{train,val,test}.jsonl` and `data/processed/self_authored.jsonl` are gitignored either way and must be uploaded separately (Step 2) — they are not in git history under either option.

In [ ]:
# Step 1a: clone from origin (requires the branch to be pushed first)
BRANCH = "feature/primary-detector-redteam"  # confirm this matches the pushed branch
!git clone --branch $BRANCH --single-branch https://github.com/KNakul242/agent-context-guardrail.git repo
%cd repo

# Step 1b (alternative, if not pushing to origin): comment out 1a above,
# uncomment below, and upload a zip of the working tree when prompted.
# from google.colab import files
# uploaded = files.upload()  # upload e.g. agent-context-guardrail.zip
# !unzip -q *.zip -d repo
# %cd repo

In [ ]:
# Step 2: upload the gitignored processed-data files (train/val/test + self_authored if present)
from google.colab import files
import os
os.makedirs("data/processed", exist_ok=True)
print("Upload train.jsonl, val.jsonl, test.jsonl (and self_authored.jsonl if you have it):")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f"data/processed/{name}")
!ls -la data/processed/

In [ ]:
# Step 3: install exact-pinned deps (requirements.txt is version-pinned, not floor-pinned --
# ISSUE-3, docs/ISSUES.md, was CAUSED by an unpinned-version default-behavior change, so a
# fresh Colab install resolving a different version than locally-verified is a real risk here,
# not a hypothetical one) and confirm GPU is actually visible to torch.
#
# ISSUE-7 (docs/ISSUES.md): Colab ships torchvision pre-installed, built against Colab's own
# default torch (2.11.0). Installing our pinned torch==2.13.0 without touching torchvision left
# it ABI-incompatible -- its compiled ops (e.g. `nms`) fail to register against the new torch
# build, and transformers==5.15.1's AutoModelForSequenceClassification resolution unconditionally
# pulls in a torchvision-dependent import chain while loading DebertaV2ForSequenceClassification
# (an apparent eager-import of object-detection loss code that has nothing to do with our text
# classifier). We use zero torchvision functionality anywhere in this project -- uninstalling it
# entirely is the fix: transformers' is_torchvision_available() checks package *presence*, not
# functional import success, so removing it lets that check correctly return False and skip the
# broken code path, rather than attempting it against a version-mismatched torchvision. Do NOT
# "fix" this by downgrading torch to Colab's stock 2.11.0 instead -- that reopens the exact
# unpinned-version-drift risk ISSUE-6's exact-pinning was added to close.
!pip install -q -r requirements.txt
!pip uninstall -y -q torchvision
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU visible -- fix Runtime -> Change runtime type -> GPU before continuing"
print("device name:", torch.cuda.get_device_name(0))

In [ ]:
# Helper: resolve "the checkpoint" for a run dir as its highest-numbered epoch_N/ subdir
# (docs/ISSUES.md ISSUE-6 correction -- each epoch has its own directory now, never a
# shared overwritten path). Used by Step 5 and Step 6b/6c below.
import re
from pathlib import Path

def latest_epoch_dir(run_dir):
    run_dir = Path(run_dir)
    epoch_dirs = [p for p in run_dir.glob("epoch_*") if p.is_dir()]
    if not epoch_dirs:
        return None
    return max(epoch_dirs, key=lambda p: int(re.match(r"epoch_(\d+)", p.name).group(1)))

## Step 4: apples-to-apples smoke test

Identical config to the local run that produced F1=0.833 / ROC-AUC=0.916 (`docs/ISSUES.md` ISSUE-3's verification): `--limit 512 --epochs 3 --lr 2e-5 --seed 6`. If Colab reproduces comparably healthy loss/F1/AUC, that's real evidence the fp32 fix generalizes across hardware rather than being an MPS-specific artifact, and gives a real per-step speed read for CUDA before committing to the full run.

In [ ]:
import glob

_before = set(glob.glob("models/smoke_test/*"))

!python scripts/train_primary.py \
  --success-criterion "Colab smoke test (D29): reproduce the local --limit 512 --lr 2e-5 --seed 6 result (F1=0.833, ROC-AUC=0.916) on CUDA to confirm the fp32 fix generalizes across hardware, and get a real per-step timing read before the full run." \
  --epochs 3 --batch-size 16 --lr 2e-5 --seed 6 --limit 512

_after = set(glob.glob("models/smoke_test/*"))
_new_dirs = list(_after - _before)
assert len(_new_dirs) == 1, f"expected exactly one new smoke_test run dir, found {_new_dirs}"
SMOKE_TEST_RUN_DIR = _new_dirs[0]
SMOKE_TEST_CHECKPOINT = latest_epoch_dir(SMOKE_TEST_RUN_DIR)
assert SMOKE_TEST_CHECKPOINT is not None, f"no epoch_N/ subdir found under {SMOKE_TEST_RUN_DIR}"
print("smoke-test checkpoint:", SMOKE_TEST_CHECKPOINT)

## Step 5: REAL gate — Step 6 (the full run) does not execute unless this cell's assertion passes

Peer-review finding, fixed here: the original version of this cell was commented out with a manual "eyeball it" instruction — nothing stopped a straight run-all from skipping the check entirely under time pressure. This is now a live assertion. Floor is deliberately loose (F1 > 0.5, meaningfully above chance, not "matches local within noise") since exact cross-hardware reproduction isn't guaranteed even with seeding fixed — see the reproducibility note at the top.

In [ ]:
import json, subprocess

F1_FLOOR = 0.5  # meaningfully above chance (~0.0 is what the pre-ISSUE-3-fix collapse looked like)

result = subprocess.run(
    ["python", "scripts/evaluate.py", "--checkpoint", str(SMOKE_TEST_CHECKPOINT), "--split", "data/processed/val.jsonl"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
report = json.loads(result.stdout)

print(f"local reference:  F1=0.833  ROC-AUC=0.916")
print(f"Colab smoke test: F1={report['f1']:.3f}  ROC-AUC={report['roc_auc']:.3f}")

assert report["f1"] > F1_FLOOR, (
    f"Colab smoke test F1={report['f1']:.3f} did not clear the {F1_FLOOR} floor -- "
    "the fp32 fix may not be generalizing to CUDA as expected (or something else is wrong). "
    "Do NOT proceed to the full run (Step 6) until this is understood -- investigate rather than re-running blindly."
)
print("\nGATE PASSED -- proceeding to Step 6 is reasonable.")

## Step 6: full run, launched in the BACKGROUND (only after Step 5's gate passes)

Peer-review finding, fixed here: the original version ran the full training as one blocking foreground cell, making "download periodically" instructions inoperable — there was no way to trigger a download while the blocking cell held the kernel. This launches training as a background subprocess instead, so Step 6b/6c below can be re-run at any time (including repeatedly, while training is still going) to check progress or download the current checkpoint — a fresh `epoch_N/` directory appears after every completed epoch (ISSUE-5/ISSUE-6) regardless of which cell is currently "active."

T4 has a hard 16GB VRAM ceiling (unlike MPS's soft/shared one). The 512-example smoke test above uses the same batch_size=16 as the full run, which is reassuring but not a guarantee — the full pool may contain a different density of near-512-token documents than the smoke test's random slice. If Step 6 OOMs on CUDA, that's a real, actionable finding (reduce batch_size), not a reason to distrust the smoke test.

In [ ]:
import subprocess, sys

full_run_log = open("full_run.log", "w")
full_run_proc = subprocess.Popen(
    [
        sys.executable, "scripts/train_primary.py",
        "--success-criterion",
        "Full run on Colab CUDA, parallel to the local MPS run (docs/DECISIONS.md D29). Same fp32-pinned, eps=1e-6, seeded pipeline as local. Success bar: training loss decreases with no NaN, validation F1/ROC-AUC comparable to or better than the local smoke test's 0.833/0.916. Full DoD metric suite + red-team bypass rate, reported honestly regardless of outcome, are what actually count.",
        "--epochs", "3", "--batch-size", "16", "--lr", "2e-5", "--seed", "0",
    ],
    stdout=full_run_log, stderr=subprocess.STDOUT,
)
FULL_RUN_DIR = "models/primary"
print(f"full run launched in background, pid={full_run_proc.pid}. Run Step 6b/6c below to check progress or download -- you do not need to wait for this cell.")

In [ ]:
# Step 6b: check progress -- safe to re-run anytime, does not disturb the background training process
import json

print("still running:", full_run_proc.poll() is None)
latest = latest_epoch_dir(FULL_RUN_DIR)
if latest is not None:
    with open(latest / "manifest.json") as f:
        print(f"latest completed checkpoint: {latest}")
        print(json.dumps(json.load(f), indent=2))
else:
    print("no epoch_N/ yet -- epoch 1 hasn't completed")
print("\n--- last 20 lines of full_run.log ---")
!tail -20 full_run.log

In [ ]:
# Step 6c: download the CURRENT latest-epoch checkpoint -- safe to re-run anytime, including while
# training is still in progress (it downloads whatever the latest completed epoch_N/ is, per
# ISSUE-5/ISSUE-6). Run this after every epoch or two, not just once at the end -- Colab's disk
# does not survive a disconnect. Zips the whole run dir (all completed epochs), not just the latest,
# so you keep every recoverable point, not only the most recent one.
!zip -rq primary_checkpoint.zip models/primary/
from google.colab import files
files.download("primary_checkpoint.zip")